# 02 - Feature Engineering
## Predicting Human Intestinal Absorption (HIA)
Mohammad Karamath Fardeen | 25251265 | MSc Data Science & Analytics, Maynooth University

This notebook demonstrates the RDKit-based molecular descriptor computation pipeline used to convert SMILES strings into the 12 features used by the classical ML models. It mirrors `src/features/compute_descriptors.py` but with inline explanation and visual sanity checks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Draw
from tqdm import tqdm

SEED = 42

## 1. Load raw SMILES data

In [ ]:
train = pd.read_csv("../data/raw/hia_train.csv")
train.head()

## 2. Visualise a few example molecules

Sanity check: confirm SMILES strings parse correctly and look at structures from both classes.

In [ ]:
high_examples = train[train["Y"] == 1]["Drug"].head(3).tolist()
low_examples  = train[train["Y"] == 0]["Drug"].head(3).tolist()

mols = [Chem.MolFromSmiles(s) for s in high_examples + low_examples]
labels = ["High absorption"]*3 + ["Low absorption"]*3

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(250, 250), legends=labels)
img

## 3. Define the descriptor computation function

Same 12 descriptors used throughout the project, matching `src/features/compute_descriptors.py`.

In [ ]:
DESCRIPTOR_FUNCTIONS = {
    "MolWt":        Descriptors.MolWt,
    "LogP":         Descriptors.MolLogP,
    "TPSA":         Descriptors.TPSA,
    "HBD":          rdMolDescriptors.CalcNumHBD,
    "HBA":          rdMolDescriptors.CalcNumHBA,
    "RotBonds":     rdMolDescriptors.CalcNumRotatableBonds,
    "RingCount":    rdMolDescriptors.CalcNumRings,
    "AromaticRings":rdMolDescriptors.CalcNumAromaticRings,
    "HeavyAtoms":   Descriptors.HeavyAtomCount,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "MolMR":        Descriptors.MolMR,
    "NumHeteroatoms": rdMolDescriptors.CalcNumHeteroatoms,
}

def smiles_to_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {name: fn(mol) for name, fn in DESCRIPTOR_FUNCTIONS.items()}

# Quick test on one molecule
example = smiles_to_descriptors(train["Drug"].iloc[0])
example

## 4. Compute descriptors for the full training set

In [ ]:
records = []
failed = 0

for _, row in tqdm(train.iterrows(), total=len(train), desc="Computing descriptors"):
    feats = smiles_to_descriptors(row["Drug"])
    if feats is None:
        failed += 1
        continue
    feats["Drug_ID"] = row["Drug_ID"]
    feats["SMILES"]  = row["Drug"]
    feats["Y"]       = int(row["Y"])
    records.append(feats)

features_df = pd.DataFrame(records)
print(f"Successfully processed: {len(records)} | Failed: {failed}")
features_df.head()

## 5. Check for missing values

In [ ]:
print("Missing values per column:")
print(features_df.isnull().sum())

## 6. Verify against pre-computed processed features

Sanity check that this notebook's output matches the saved `data/processed/` files used by the modelling scripts.

In [ ]:
saved_features = pd.read_csv("../data/processed/hia_train_features.csv")

FEATURE_COLS = list(DESCRIPTOR_FUNCTIONS.keys())
match = np.allclose(
    features_df.sort_values("Drug_ID")[FEATURE_COLS].values,
    saved_features.sort_values("Drug_ID")[FEATURE_COLS].values,
    equal_nan=True
)
print(f"Notebook-computed features match saved processed features: {match}")

## Summary

This notebook reproduces the feature engineering step end-to-end: every SMILES string in the training set was successfully parsed by RDKit (0 failures), 12 molecular descriptors were computed per molecule, and the resulting feature matrix matches the pre-computed `data/processed/` files used by the downstream classical ML training scripts. These same 12 features were later ranked by SHAP importance in the explainability analysis.